# 3.8 世界模型 (World Models)

> 🕐 预估学习时间：35分钟

世界模型（World Models）是智能体学习环境动态规律的统一框架，让模型不仅能“感知”世界，还能“想象”未来。

近年来，Sora、Genie、DreamerV3 等系统展示了世界模型在视频生成、智能体规划、强化学习中的巨大潜力。LeCun 提出的 JEPA 架构则指明了在表示空间而非像素空间建模世界动态的方向。

本节涵盖：
- 世界模型的基本框架
- JEPA 联合嵌入预测架构
- DreamerV3 风格的循环状态空间模型 RSSM
- LLM 作为隐式世界模型
- 视频生成与世界模拟

## 1. 世界模型概述

**什么是世界模型**：
- 学习环境的动态规律：给定当前状态 $s_t$ 和动作 $a_t$，预测下一状态 $s_{t+1}$
- 内部维护对世界的“心理模型”，可在脑中 rollout 未来轨迹
- 是规划（planning）、模拟（simulation）、想象（imagination）的基础

**为什么重要**：
- **规划**：在模型中搜索最优动作序列，避免真实环境试错
- **数据效率**：在想象空间中训练，大幅减少真实交互需求
- **视频生成**：Sora 等模型本质是学习物理世界动态的生成模型
- **具身智能**：机器人需要在脑中预测动作后果才能精细操作

**代表系统**：
- **Sora**：OpenAI 的视频生成模型，被视为“世界模拟器”
- **Genie**：DeepMind 的可交互环境生成世界模型
- **DreamerV3**：基于 RSSM 的通用强化学习世界模型
- **JEPA**：LeCun 提出的联合嵌入预测架构

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math

torch.manual_seed(42)


# 世界模型基类：预测下一状态
class WorldModel(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=64):
        super().__init__()
        self.transition = nn.GRUCell(state_dim + action_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, state_dim)

    def forward(self, state, action, hidden=None):
        x = torch.cat([state, action], dim=-1)
        hidden = self.transition(x, hidden)
        next_state = self.head(hidden)
        return next_state, hidden

    def rollout(self, init_state, actions, horizon):
        # 在想象空间中展开 H 步轨迹
        state = init_state
        hidden = None
        trajectory = [state]
        for t in range(horizon):
            state, hidden = self.forward(state, actions[t], hidden)
            trajectory.append(state)
        return torch.stack(trajectory, dim=0)


# 简单的网格世界环境
class EnvironmentSimulator:
    def __init__(self, size=5):
        self.size = size
        self.pos = np.array([0, 0])
        self.goal = np.array([size - 1, size - 1])

    def reset(self):
        self.pos = np.array([0, 0])
        return self._obs()

    def step(self, action):
        # 0:up 1:down 2:left 3:right
        moves = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
        dx, dy = moves[action]
        self.pos = np.clip(self.pos + np.array([dx, dy]), 0, self.size - 1)
        reward = 1.0 if np.all(self.pos == self.goal) else 0.0
        return self._obs(), reward

    def _obs(self):
        obs = np.zeros(self.size * self.size)
        obs[self.pos[0] * self.size + self.pos[1]] = 1.0
        return obs.astype(np.float32)


state_dim = 25
action_dim = 4
model = WorldModel(state_dim, action_dim, hidden_dim=64)
env = EnvironmentSimulator(size=5)

print('=== World Model Framework ===')
state = torch.tensor(env.reset()).unsqueeze(0)
print(f'Initial state shape: {state.shape}')

actions = [F.one_hot(torch.tensor(a), action_dim).float().unsqueeze(0) for a in [3, 3, 1, 1, 3]]
traj = model.rollout(state, actions, horizon=len(actions))
print(f'Predicted trajectory shape: {traj.shape}')
print(f'Rollout horizon: {len(actions)} steps')

obs, reward = env.step(3)
print(f'\nReal env step reward: {reward}')

print(f'\nKey: World models learn environment dynamics to enable planning and imagination.')

## 2. JEPA 架构

**Joint Embedding Predictive Architecture (JEPA)** 由 LeCun 提出，是自监督学习世界模型的核心思想。

**核心思想**：
- 不在像素空间预测未来（pixel-level prediction 噪声大、细节多）
- 而在**表示空间（latent space）**预测未来
- 通过编码器将观测映射到抽象表示，再在表示空间做预测

**架构组件**：
1. **Context encoder** $s_x = \phi_x(x)$：编码可见上下文
2. **Target encoder** $s_y = \phi_y(y)$：编码目标（未来）观测
3. **Predictor** $\hat{s}_y = p(s_x, a)$：在表示空间预测未来

**优势**：
- **抽象**：忽略无关像素细节，关注高层语义
- **稳定**：避免像素级预测的“模糊平均”问题
- **可扩展**：可堆叠多层级 JEPA 形成层次化世界模型

**关键挑战**：表示坍塌（representation collapse）—— 需通过 stop-gradient、EMA encoder、能量距离等技巧防止

In [ ]:
# 在表示空间预测未来状态
class LatentPredictor(nn.Module):
    def __init__(self, latent_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim + action_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, latent, action):
        return self.net(torch.cat([latent, action], dim=-1))


# JEPA：在表示空间预测未来
class JEPAWorldModel(nn.Module):
    def __init__(self, obs_dim, action_dim, latent_dim=32):
        super().__init__()
        self.context_encoder = nn.Sequential(
            nn.Linear(obs_dim, 128), nn.GELU(),
            nn.Linear(128, latent_dim),
        )
        # target encoder 使用 EMA 更新
        self.target_encoder = nn.Sequential(
            nn.Linear(obs_dim, 128), nn.GELU(),
            nn.Linear(128, latent_dim),
        )
        for p in self.target_encoder.parameters():
            p.requires_grad = False
        self.predictor = LatentPredictor(latent_dim, action_dim, hidden_dim=128)

    @torch.no_grad()
    def update_target_encoder(self, momentum=0.99):
        for p, p_target in zip(self.context_encoder.parameters(),
                               self.target_encoder.parameters()):
            p_target.data.mul_(momentum).add_(p.data, alpha=1 - momentum)

    def forward(self, obs_t, obs_next, action):
        s_x = self.context_encoder(obs_t)
        with torch.no_grad():
            s_y = self.target_encoder(obs_next)
        s_y_pred = self.predictor(s_x, action)
        # 使用 cosine + MSE 防止表示坍塌
        loss_mse = F.mse_loss(s_y_pred, s_y)
        loss_cos = 1 - F.cosine_similarity(s_y_pred, s_y, dim=-1).mean()
        return loss_mse + 0.5 * loss_cos, s_x, s_y


torch.manual_seed(42)
obs_dim, action_dim, latent_dim = 25, 4, 32
jepa = JEPAWorldModel(obs_dim, action_dim, latent_dim)
optimizer = torch.optim.AdamW(jepa.parameters(), lr=1e-3)

print('=== JEPA World Model ===')
for epoch in range(30):
    obs_t = torch.randn(16, obs_dim)
    obs_next = torch.randn(16, obs_dim)
    action = F.one_hot(torch.randint(0, action_dim, (16,)), action_dim).float()
    loss, s_x, s_y = jepa(obs_t, obs_next, action)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    jepa.update_target_encoder()
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}: loss={loss.item():.4f}')

print(f'\nContext latent shape: {s_x.shape}')
print(f'Target latent shape: {s_y.shape}')

print(f'\nKey: JEPA predicts future in latent space, avoiding pixel-level noise.')

## 3. DreamerV3 风格世界模型

**DreamerV3** 是基于循环状态空间模型（RSSM）的通用强化学习算法，在 150+ 任务上单套超参即可超越人类。

**RSSM (Recurrent State-Space Model)**：
- 结合**确定性循环状态** $h_t$（GRU）和**随机隐状态** $z_t$
- 确定性路径保证长期记忆，随机路径建模环境不确定性
- 状态转移：$h_t = f(h_{t-1}, z_{t-1}, a_{t-1})$
- 随机状态：$z_t \sim q(z_t | h_t, o_t)$（后验）或 $\hat{z}_t \sim p(z_t | h_t)$（先验）

**Dreamer 训练流程**：
1. **学习世界模型**：在真实轨迹上训练 RSSM + decoder + reward head
2. **想象 rollout**：在学到的世界模型中想象未来轨迹
3. **Actor-Critic**：在想象空间中优化策略和价值函数

**优势**：
- 高数据效率（在想象中训练）
- 通用性强（同一套超参适用多任务）
- 可处理部分可观测、随机环境

In [ ]:
# Recurrent State-Space Model
class RSSM(nn.Module):
    def __init__(self, action_dim, stoch_dim=16, hidden_dim=64, obs_dim=25):
        super().__init__()
        self.stoch_dim = stoch_dim
        self.hidden_dim = hidden_dim
        # 确定性循环
        self.cell = nn.GRUCell(action_dim + stoch_dim, hidden_dim)
        # 先验 p(z_t | h_t)
        self.prior_net = nn.Sequential(
            nn.Linear(hidden_dim, 32), nn.GELU(),
            nn.Linear(32, 2 * stoch_dim),
        )
        # 后验 q(z_t | h_t, o_t)
        self.post_net = nn.Sequential(
            nn.Linear(hidden_dim + obs_dim, 32), nn.GELU(),
            nn.Linear(32, 2 * stoch_dim),
        )

    def initial_state(self, batch_size):
        return (torch.zeros(batch_size, self.hidden_dim),
                torch.zeros(batch_size, self.stoch_dim))

    def step(self, prev_state, prev_action, obs=None):
        h_prev, z_prev = prev_state
        x = torch.cat([z_prev, prev_action], dim=-1)
        h = self.cell(x, h_prev)
        if obs is not None:
            # 训练时使用后验
            post = self.post_net(torch.cat([h, obs], dim=-1))
            mean, std = post.chunk(2, dim=-1)
            std = F.softplus(std) + 0.1
            z = mean + std * torch.randn_like(std)
        else:
            # 想象时使用先验
            prior = self.prior_net(h)
            mean, std = prior.chunk(2, dim=-1)
            std = F.softplus(std) + 0.1
            z = mean + std * torch.randn_like(std)
        return (h, z), (mean, std)


# DreamerV3 风格世界模型
class DreamerWorldModel(nn.Module):
    def __init__(self, obs_dim, action_dim, stoch_dim=16, hidden_dim=64):
        super().__init__()
        self.rssm = RSSM(action_dim, stoch_dim, hidden_dim, obs_dim)
        feat_dim = hidden_dim + stoch_dim
        self.decoder = nn.Sequential(
            nn.Linear(feat_dim, 64), nn.GELU(),
            nn.Linear(64, obs_dim),
        )
        self.reward_head = nn.Sequential(
            nn.Linear(feat_dim, 32), nn.GELU(),
            nn.Linear(32, 1),
        )

    def rollout(self, obs_seq, actions):
        # 训练时 rollout：使用后验
        T, B = obs_seq.shape[0], obs_seq.shape[1]
        state = self.rssm.initial_state(B)
        feats, rewards, recons = [], [], []
        for t in range(T):
            state, _ = self.rssm.step(state, actions[t], obs=obs_seq[t])
            h, z = state
            feat = torch.cat([h, z], dim=-1)
            feats.append(feat)
            rewards.append(self.reward_head(feat))
            recons.append(self.decoder(feat))
        return torch.stack(feats), torch.stack(rewards), torch.stack(recons)


torch.manual_seed(42)
obs_dim, action_dim = 25, 4
dreamer = DreamerWorldModel(obs_dim, action_dim)
optimizer = torch.optim.AdamW(dreamer.parameters(), lr=1e-3)

print('=== DreamerV3-style World Model ===')
T, B = 8, 4
obs_seq = torch.randn(T, B, obs_dim)
actions = F.one_hot(
    torch.randint(0, action_dim, (T, B)), action_dim
).float()

for epoch in range(20):
    feats, rewards, recons = dreamer.rollout(obs_seq, actions)
    loss = F.mse_loss(recons, obs_seq) + 0.1 * rewards.pow(2).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}: loss={loss.item():.4f}')

# 想象 rollout（无观测）
print('\nImagination rollout (no observation):')
state = dreamer.rssm.initial_state(B)
for t in range(3):
    state, _ = dreamer.rssm.step(state, actions[t], obs=None)
    h, z = state
    print(f'  Step {t}: hidden norm={h.norm(dim=-1).mean():.3f}, stoch norm={z.norm(dim=-1).mean():.3f}')

print(f'\nKey: RSSM combines deterministic recurrence with stochastic latents for imagination.')

## 4. LLM 作为世界模型

**LLM 是隐式世界模型**：
- 训练语料本身包含世界的规律（物理、社会、逻辑）
- LLM 通过 next-token prediction 隐式学到了这些规律
- 上下文学习（in-context learning）可视为一种“模拟”

**作为世界模型的使用方式**：
- **环境模拟器**：用 LLM 模拟文本环境响应（如 TextWorld）
- **状态预测**：给定当前状态描述，让 LLM 预测下一状态
- **动作评估**：让 LLM 评估某动作的后果

**局限性**：
- **非确定性**：采样温度导致结果不稳定
- **幻觉**：可能生成违反物理规律的“未来”
- **长程一致性**：长 rollout 后状态会漂移
- **缺乏 grounding**：文本与真实物理世界脱节

**研究方向**：
- 用 RLHF / RLAIF 校准世界模型
- 多模态 LLM（VLM）补足视觉 grounding
- 工具调用增强（调用物理引擎、计算器）

In [ ]:
# 用 LLM（这里用简化模拟）作为世界模型
class LLMWorldModel(nn.Module):
    def __init__(self, vocab_size=50, hidden_dim=64, seed=42):
        super().__init__()
        torch.manual_seed(seed)
        self.embed = nn.Embedding(vocab_size, hidden_dim)
        self.head = nn.Linear(hidden_dim, vocab_size)
        self.vocab_size = vocab_size
        # 简单的“环境知识”表
        self.transition_table = {
            ('go_north', 'room_a'): 'room_b',
            ('go_south', 'room_b'): 'room_a',
            ('go_east', 'room_b'): 'room_c',
            ('go_west', 'room_c'): 'room_b',
        }

    def simulate(self, state, action):
        # 模拟环境响应
        key = (action, state)
        if key in self.transition_table:
            next_state = self.transition_table[key]
            reward = 1.0 if next_state == 'room_c' else 0.0
            return next_state, reward, f'You move {action.split(chr(95))[1]} to {next_state}.'
        return state, 0.0, f'You cannot go that way from {state}.'

    def predict_token(self, context_tokens):
        # 模拟 LLM 的 next-token 预测
        embeds = self.embed(context_tokens)
        logits = self.head(embeds.mean(dim=0, keepdim=True))
        return logits.argmax(dim=-1)


print('=== LLM as World Model ===')
llm_wm = LLMWorldModel()

print('Text environment simulation:')
state = 'room_a'
for action in ['go_north', 'go_east', 'go_west']:
    next_state, reward, desc = llm_wm.simulate(state, action)
    print(f'  [{state}] --{action}--> [{next_state}] reward={reward:.1f}')
    print(f'    {desc}')
    state = next_state

# 对比显式 vs LLM 世界模型
print('\nExplicit vs LLM world model comparison:')
print('  Explicit: deterministic, fast, requires training data')
print('  LLM-based: flexible, language-grounded, may hallucinate')

context = torch.randint(0, 50, (5,))
pred = llm_wm.predict_token(context)
print(f'\nLLM next-token prediction: {pred.item()}')

print(f'\nKey: LLMs serve as implicit world models via in-context simulation.')

## 5. 视频生成与世界模拟

**Sora 与视频世界模型**：
- OpenAI 将 Sora 描述为“世界模拟器”（world simulator）
- 视频生成模型不仅生成像素，更学习物理规律（重力、碰撞、遮挡）
- 长视频生成需要一致的世界状态维护

**视频世界模型的关键组件**：
- **时空注意力**：同时建模空间相关性和时间动态
- **因果时间掩码**：防止未来信息泄漏
- **分层时间尺度**：粗粒度全局动态 + 细粒度局部变化
- **物理一致性**：保持物体持久性、运动连续性

**挑战**：
- 长视频的时间一致性
- 复杂物理现象（流体、刚体、柔体）
- 多物体交互与遮挡
- 计算成本（高分辨率长视频）

In [ ]:
# 沿时间维度的自注意力
class TemporalAttention(nn.Module):
    def __init__(self, dim, n_heads=4):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        # x: (B, T, D)
        B, T, D = x.shape
        # 因果掩码：只能看过去帧
        mask = torch.triu(torch.full((T, T), float('-inf')), diagonal=1)
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = [t.transpose(1, 2) for t in (q, k, v)]
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = attn + mask
        attn = F.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T, D)
        return self.proj(out)


# 视频世界模型：预测未来帧
class VideoWorldModel(nn.Module):
    def __init__(self, frame_dim=64, hidden_dim=128, n_heads=4):
        super().__init__()
        self.frame_embed = nn.Linear(frame_dim, hidden_dim)
        self.temporal_attn = TemporalAttention(hidden_dim, n_heads)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.GELU(),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.frame_head = nn.Linear(hidden_dim, frame_dim)

    def forward(self, frames):
        # frames: (B, T, frame_dim)
        x = self.frame_embed(frames)
        x = x + self.temporal_attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return self.frame_head(x)

    def predict_next(self, frames, n_steps=1):
        # 自回归预测未来 n 帧
        cur = frames
        preds = []
        for _ in range(n_steps):
            pred = self.forward(cur)
            next_frame = pred[:, -1:, :]
            preds.append(next_frame)
            cur = torch.cat([cur, next_frame], dim=1)
        return torch.cat(preds, dim=1)


torch.manual_seed(42)
frame_dim, hidden_dim = 64, 128
video_wm = VideoWorldModel(frame_dim, hidden_dim)
optimizer = torch.optim.AdamW(video_wm.parameters(), lr=1e-3)

print('=== Video World Model ===')
B, T = 4, 8
frames = torch.randn(B, T, frame_dim)

for epoch in range(20):
    pred = video_wm(frames[:, :-1])
    target = frames[:, 1:]
    loss = F.mse_loss(pred, target)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}: loss={loss.item():.4f}')

future = video_wm.predict_next(frames, n_steps=3)
print(f'\nInput frames shape: {frames.shape}')
print(f'Predicted future frames shape: {future.shape}')
print(f'Per-frame prediction error: {(future - frames[:, -3:]).pow(2).mean():.4f}')

print(f'\nKey: Video world models learn spatiotemporal dynamics for frame prediction.')

## 📝 课后思考题

1. JEPA 为什么选择在表示空间而非像素空间预测？这种设计如何避免表示坍塌？
2. RSSM 中确定性状态 $h_t$ 与随机状态 $z_t$ 各自的作用是什么？为什么需要两者结合？
3. LLM 作为世界模型存在哪些根本性局限？多模态能否解决这些问题？
4. Sora 等视频生成模型在何种意义上是“世界模拟器”？它们真正理解物理规律吗？